In [3]:
import torch
import requests
import json
import base64
import os
import warnings
warnings.filterwarnings("ignore")

# ====================== Core Configuration ======================
POE_API_KEY = "frQ4l61J3rbV7VNaKY-4sPs7a-KgmLabkoTdTo-NkBI"
POE_BASE_URL = "https://api.poe.com/v1"
POE_CHAT_COMPLETIONS_URL = f"{POE_BASE_URL}/chat/completions"
POE_MODEL = "gpt-4o-mini"

IMAGE_FOLDER_PATH = "/kaggle/input/datasets/snnn9017/animal-images"
SUPPORTED_FORMATS = [".jpg", ".jpeg", ".png"]
JSON_SAVE_PATH = "./batch_sticker_prompts.json"

MODEL_ID = "sd-legacy/stable-diffusion-v1-5"
DEVICE = "cuda"
TORCH_DTYPE = torch.float16

# ====================== Tool Functions ======================
def save_prompt_to_json(prompt_data_list):
    if os.path.exists(JSON_SAVE_PATH):
        with open(JSON_SAVE_PATH, "r", encoding="utf-8") as f:
            try:
                existing_data = json.load(f)
                existing_data = existing_data if isinstance(existing_data, list) else []
            except json.JSONDecodeError:
                existing_data = []
    else:
        existing_data = []

    existing_data.extend(prompt_data_list)

    with open(JSON_SAVE_PATH, "w", encoding="utf-8") as f:
        json.dump(existing_data, f, indent=4, ensure_ascii=False)

    print(f"✅ Batch prompt save completed! Total {len(prompt_data_list)} entries.")

def encode_image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def get_all_animal_images(folder_path):
    image_paths = []
    for file_name in os.listdir(folder_path):
        file_ext = os.path.splitext(file_name)[1].lower()
        if file_ext in SUPPORTED_FORMATS:
            full_path = os.path.join(folder_path, file_name)
            image_paths.append(full_path)
    return image_paths

# ====================== Core Function ======================
def generate_prompt_for_single_image(image_path):
    try:
        base64_image = encode_image_to_base64(image_path)
    except Exception as e:
        print(f"❌ Image encoding failed {image_path}: {e}")
        return None

    system_prompt = """
    You are a professional Stable Diffusion prompt expert.
    STRICT RULES:
    1. INSERT "sks" RIGHT BEFORE THE ANIMAL NAME.
    2. Example: A fluffy white Pomeranian sks dog.
    3.Describe the animal's **posture/gesture** clearly (sitting, standing, lying, running, looking at camera, facing side, etc.)
    4. DO NOT describe background, environment, or scene at all..
    5. ONLY use "sticker style" as the style. DO NOT use cartoon, watercolor, anime, painting, or any other style words.
    6. Only describe the animal + posture + sticker style.

    Output ONLY in this format:
    Final Prompt: [your prompt]
    """

    headers = {
        "Authorization": f"Bearer {POE_API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": POE_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}},
                    {"type": "text", "text": "Generate prompt"}
                ]
            }
        ],
        "max_tokens": 100,
        "temperature": 0.6
    }

    try:
        response = requests.post(POE_CHAT_COMPLETIONS_URL, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        result = response.json()
    except Exception as e:
        print(f"❌ API call failed {image_path}: {e}")
        return None

    content = result["choices"][0]["message"]["content"].strip()
    try:
        sd_prompt = content.split("Final Prompt:")[-1].strip()
    except:
        print(f"❌ Parse error: {content}")
        return None

    prompt_data = {
        "file_name": os.path.basename(image_path),
        "sd_sticker_prompt": sd_prompt
    }

    print(f"✅ Success: {sd_prompt[:90]}...")
    return prompt_data

# ====================== Batch ======================
def batch_generate_prompts():
    image_paths = get_all_animal_images(IMAGE_FOLDER_PATH)
    if not image_paths:
        print(f"❌ No valid images!")
        return

    print(f"📁 Found {len(image_paths)} images...")
    prompt_data_list = []

    for idx, image_path in enumerate(image_paths, 1):
        print(f"\n[{idx}/{len(image_paths)}] Processing: {os.path.basename(image_path)}")
        prompt_data = generate_prompt_for_single_image(image_path)
        if prompt_data:
            prompt_data_list.append(prompt_data)

    if prompt_data_list:
        save_prompt_to_json(prompt_data_list)
    else:
        print("❌ No valid prompts!")

# ====================== Run ======================
if __name__ == "__main__":
    if not os.path.exists(IMAGE_FOLDER_PATH):
        os.makedirs(IMAGE_FOLDER_PATH)
        print(f"⚠️ Folder created")
    else:
        batch_generate_prompts()

📁 Found 37 images...

[1/37] Processing: animal_2.jpeg
✅ Success: Two sks cats, one sitting and looking at the camera while the other is lying down, sticker...

[2/37] Processing: animal_26.jpeg
✅ Success: A fluffy black and white persian sks cat sitting with its eyes closed, in sticker style....

[3/37] Processing: animal_5.jpeg
✅ Success: sks penguin standing sticker style....

[4/37] Processing: animal_6.jpeg
✅ Success: A fluffy white sks rabbit sitting and looking at the camera, sticker style....

[5/37] Processing: animal_28.jpeg
✅ Success: A sitting sks cat in sticker style....

[6/37] Processing: animal_29.jpeg
✅ Success: A calm sks cat lying down in a relaxed posture, sticker style....

[7/37] Processing: animal_15.jpeg
✅ Success: A curious gray sks cat looking at the camera with its eyes wide open, sticker style....

[8/37] Processing: animal_36.jpeg
✅ Success: A curious sks cat, looking at camera, sticker style....

[9/37] Processing: animal_32.jpeg
✅ Success: A sleek grey sk